In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time
import logging
import sys
import json
import copy
from datetime import datetime

# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_cifar_polynomial.log'):
    """设置日志记录"""
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    
    # 文件处理器
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    
    # 控制台处理器
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

# 设置随机种子和设备
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. 数据加载与预处理 (保持不变) ---
def get_cifar10_loaders(batch_size=128):
    """获取CIFAR-10数据加载器"""
    logging.info("正在准备CIFAR-10数据集...")
    
    # 数据预处理
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # 训练集
    trainset = torchvision.datasets.CIFAR10(
        root='/home/zxc/CodeBase/cofrnet/data', 
        train=True, 
        download=True, 
        transform=transform
    )
    trainloader = torch.utils.data.DataLoader(
        trainset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=2
    )
    
    # 测试集
    testset = torchvision.datasets.CIFAR10(
        root='/home/zxc/CodeBase/cofrnet/data', 
        train=False, 
        download=True, 
        transform=transform
    )
    testloader = torch.utils.data.DataLoader(
        testset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2
    )
    
    logging.info("数据集准备完毕。")
    return trainloader, testloader

# --- 3. 模型架构定义 (使用cfnet_classifier中的架构) ---

class PolynomialTerm(nn.Module):
    """多项式项模块"""
    def __init__(self, input_dim, output_dim, degree):
        super().__init__()
        self.degree = degree
        self.projection = nn.Linear(input_dim, output_dim)
        self.coeffs = nn.Parameter(torch.randn(output_dim, degree + 1) * 0.1)

    def forward(self, x):
        z = torch.tanh(self.projection(x))
        powers = [torch.ones_like(z)]
        for d in range(1, self.degree + 1):
            powers.append(powers[-1] * z)
        z_powered = torch.stack(powers, dim=-1)
        output = torch.sum(z_powered * self.coeffs, dim=2)
        return output

class CFNet(nn.Module):
    """CFNet网络"""
    def __init__(self, input_dim, output_dim, depth, poly_degree):
        super().__init__()
        self.output_dim = output_dim
        self.terms = nn.ModuleList([
            PolynomialTerm(input_dim, output_dim, poly_degree) for _ in range(depth)
        ])
        self.raw_betas = nn.ParameterList([
            nn.Parameter(torch.full((output_dim,), 0.1)) for _ in range(depth - 1)
        ])

    def forward(self, x):
        if not self.terms:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        output = F.softplus(self.terms[-1](x)) + 1.0
        for i in range(len(self.terms) - 2, -1, -1):
            beta = F.softplus(self.raw_betas[i])
            output = self.terms[i](x) + beta / output
        return output

class EnsembleResCoFrNet(nn.Module):
    """集成模型：支持候选模型的选择性加入机制"""
    def __init__(self, input_dim, output_dim, shallow_depth, poly_degree, learning_rate):
        super().__init__()
        self.models = nn.ModuleList()          # 已采纳的正式模型
        self.candidate_model = None           # 正在评估的候选模型
        self.learning_rate = learning_rate
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.shallow_depth = shallow_depth
        self.poly_degree = poly_degree

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        if not self.models and self.candidate_model is None:
            return torch.zeros(x.shape[0], self.output_dim, device=x.device)
        total_output = torch.zeros(x.shape[0], self.output_dim, device=x.device)
        for model in self.models:
            total_output += self.learning_rate * model(x)
        if self.candidate_model is not None:
            total_output += self.learning_rate * self.candidate_model(x)
        return total_output

    def create_candidate(self):
        """创建一个新的候选模型用于训练"""
        self.candidate_model = CFNet(
            input_dim=self.input_dim,
            output_dim=self.output_dim,
            depth=self.shallow_depth,
            poly_degree=self.poly_degree
        )
        return self.candidate_model

    def promote_candidate(self):
        """若候选模型有效，则将其采纳为正式模型"""
        if self.candidate_model is not None:
            self.models.append(self.candidate_model)
            self.candidate_model = None
    
    def discard_candidate(self):
        """丢弃无效的候选模型"""
        self.candidate_model = None

    def freeze_for_candidate_training(self):
        """冻结所有正式模型，只训练候选模型"""
        for model in self.models:
            for param in model.parameters():
                param.requires_grad = False
        if self.candidate_model is not None:
            for param in self.candidate_model.parameters():
                param.requires_grad = True

class CoFrNetClassifier:
    """CoFrNet分类器"""
    def __init__(self, hparams):
        self.hparams = hparams
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.ensemble_model = EnsembleResCoFrNet(
            input_dim=self.hparams['input_dim'],
            output_dim=self.hparams['output_dim'],
            shallow_depth=self.hparams['shallow_depth_per_cofrnet'],
            poly_degree=self.hparams['polynomial_degree'],
            learning_rate=self.hparams['boosting_learning_rate']
        ).to(self.device)
        
        self.best_ensemble_state = None
        self.best_test_acc = 0.0
        self.best_model_size = 0
        logging.info(f"CoFrNetClassifier 初始化完毕，将在 {self.device} 上运行。")
        
    def train(self, trainloader, testloader, log_filepath, model_save_path):
        """训练模型"""
        criterion = nn.CrossEntropyLoss()
        patience = self.hparams.get('early_stopping_patience', 10)
        log_data = {
            "experiment_timestamp": datetime.now().isoformat(), 
            "hyperparameters": self.hparams, 
            "results_per_submodel": []
        }

        logging.info("="*30)
        logging.info(f"  开始 CIFAR-10 分类任务训练 (选择性加入机制)")
        logging.info("="*30)
        
        for attempt in range(self.hparams['num_models_max']):
            logging.info(f"--- 尝试添加第 {len(self.ensemble_model.models) + 1} 个子模型 (总尝试次数: {attempt + 1}/{self.hparams['num_models_max']}) ---")
            
            # 1. 创建并训练一个候选模型
            candidate_model = self.ensemble_model.create_candidate().to(self.device)
            self.ensemble_model.freeze_for_candidate_training()

            optimizer = optim.Adam(candidate_model.parameters(), 
                                 lr=self.hparams['learning_rate_adam'], 
                                 weight_decay=self.hparams['weight_decay'])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.hparams['epochs_per_model'])

            epochs_no_improve = 0
            best_val_acc_for_candidate = 0.0
            best_candidate_state_dict = None

            for epoch in range(self.hparams['epochs_per_model']):
                self.ensemble_model.train()
                running_loss = 0.0
                
                # 训练循环
                for inputs, targets in trainloader:
                    inputs, targets = inputs.view(inputs.size(0), -1).to(self.device), targets.to(self.device)
                    optimizer.zero_grad()
                    outputs = self.ensemble_model(inputs)
                    loss = criterion(outputs, targets)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item()
                
                scheduler.step()
                
                # 评估当前模型
                val_acc = self.evaluate(testloader)  # 使用测试集作为验证集
                
                if (epoch + 1) % 10 == 0:
                    logging.info(f"    Epoch {epoch+1:03d} | 训练损失: {running_loss/len(trainloader):.4f} | Val Acc: {val_acc:.2f}%")

                if val_acc > best_val_acc_for_candidate:
                    best_val_acc_for_candidate = val_acc
                    epochs_no_improve = 0
                    best_candidate_state_dict = copy.deepcopy(candidate_model.state_dict())
                else:
                    epochs_no_improve += 1
                
                if epochs_no_improve >= patience:
                    logging.info(f"--- 候选模型早停触发！---")
                    if best_candidate_state_dict:
                        candidate_model.load_state_dict(best_candidate_state_dict)
                    break
            
            # 2. 评估候选模型加入后的效果
            logging.info("候选模型训练完毕，正在评估其对整体性能的贡献...")
            test_acc_with_candidate = self.evaluate(testloader)
            logging.info(f"加入候选模型后，测试集 Acc 为: {test_acc_with_candidate:.2f}%")
            logging.info(f"当前最佳模型的测试集 Acc 为: {self.best_test_acc:.2f}%")

            # 3. 决策：是否采纳
            if test_acc_with_candidate > self.best_test_acc:
                self.best_test_acc = test_acc_with_candidate
                self.ensemble_model.promote_candidate()
                self.best_model_size = len(self.ensemble_model.models)
                self.best_ensemble_state = copy.deepcopy(self.ensemble_model.state_dict())
                logging.info(f"*** 性能提升！采纳新模型。当前模型大小: {self.best_model_size}, 新的最佳 Acc: {self.best_test_acc:.2f}% ***")
            else:
                self.ensemble_model.discard_candidate()
                logging.info(f"--- 性能未提升。丢弃该候选模型，继续尝试。---")
            
            # 记录日志
            submodel_log = {
                "attempt": attempt + 1, 
                "accepted": test_acc_with_candidate > self.best_test_acc, 
                "current_best_acc": self.best_test_acc
            }
            log_data["results_per_submodel"].append(submodel_log)
            with open(log_filepath, 'w') as f: 
                json.dump(log_data, f, indent=4)

        # 训练结束后保存最佳模型
        if self.best_ensemble_state:
            logging.info(f"训练结束。最佳模型大小: {self.best_model_size}, Acc: {self.best_test_acc:.2f}%")
            self.ensemble_model.load_state_dict(self.best_ensemble_state)
            self.save_model(model_save_path)
        else: 
            logging.error("没有训练出有效模型，无需保存。")

        return self.best_test_acc

    def evaluate(self, dataloader):
        """评估模型性能"""
        self.ensemble_model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, targets in dataloader:
                inputs, targets = inputs.view(inputs.size(0), -1).to(self.device), targets.to(self.device)
                outputs = self.ensemble_model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        accuracy = 100 * correct / total
        return accuracy

    def predict(self, inputs):
        """预测函数"""
        if self.best_ensemble_state:
            self.ensemble_model.load_state_dict(self.best_ensemble_state)
        self.ensemble_model.discard_candidate()
        self.ensemble_model.eval()
        inputs = inputs.view(inputs.size(0), -1).to(self.device)
        with torch.no_grad():
            outputs = self.ensemble_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
        return predicted.cpu().numpy()

    def save_model(self, path):
        """保存模型"""
        if self.best_ensemble_state is None:
            logging.warning("没有可保存的最佳模型状态。")
            return
        
        self.ensemble_model.discard_candidate()
        self.ensemble_model.load_state_dict(self.best_ensemble_state)

        torch.save({
            'model_state_dict': self.best_ensemble_state,
            'best_acc': self.best_test_acc,
            'best_model_size': self.best_model_size,
            'hyperparameters': self.hparams
        }, path)
        logging.info(f"最佳模型已保存到 {path}")

    @classmethod
    def load_model(cls, path):
        """加载模型"""
        checkpoint = torch.load(path, map_location=lambda storage, loc: storage)
        hparams = checkpoint['hyperparameters']
        loaded_classifier = cls(hparams)
        
        loaded_classifier.ensemble_model.discard_candidate()
        loaded_classifier.ensemble_model.models = nn.ModuleList()

        # 使用临时实例加载状态字典
        temp_ensemble = EnsembleResCoFrNet(
            input_dim=loaded_classifier.ensemble_model.input_dim,
            output_dim=loaded_classifier.ensemble_model.output_dim,
            shallow_depth=loaded_classifier.ensemble_model.shallow_depth,
            poly_degree=loaded_classifier.ensemble_model.poly_degree,
            learning_rate=loaded_classifier.ensemble_model.learning_rate
        )
        temp_ensemble.load_state_dict(checkpoint['model_state_dict'])

        # 将加载好的模型列表赋给新实例
        loaded_classifier.ensemble_model.models = temp_ensemble.models
        loaded_classifier.ensemble_model.to(loaded_classifier.device)
        
        loaded_classifier.best_ensemble_state = checkpoint['model_state_dict']
        loaded_classifier.best_test_acc = checkpoint['best_acc']
        loaded_classifier.best_model_size = checkpoint['best_model_size']
        
        logging.info(f"模型从 {path} 加载成功。大小: {checkpoint['best_model_size']}, Acc: {checkpoint['best_acc']:.2f}%")
        return loaded_classifier

# --- 4. 主执行模块 ---
if __name__ == '__main__':
    # 文件与超参数定义
    LOG_FILE = 'training_cifar_polynomial.log'
    JSON_LOG_FILE = 'training_cifar_polynomial_log.json'
    MODEL_SAVE_PATH = 'cifar_res_cofrnet_polynomial.pth'
    
    HYPERPARAMETERS = {
        "input_dim": 3 * 32 * 32,
        "output_dim": 10,
        "num_models_max": 100,
        "accuracy_target": 1.0,  # 设定一个现实的目标
        "epochs_per_model": 30,
        "shallow_depth_per_cofrnet": 2,
        "polynomial_degree": 3,  # 多项式最高次项
        "boosting_learning_rate": 0.1,
        "learning_rate_adam": 1e-4,
        "weight_decay": 1e-5,
        "early_stopping_patience": 10,
        "task_name": "CIFAR-10 Classification"
    }

    # 设置日志
    setup_logging(LOG_FILE)
    logging.info(f"实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    # 获取数据加载器
    trainloader, testloader = get_cifar10_loaders()
    
    # 创建分类器实例
    classifier = CoFrNetClassifier(HYPERPARAMETERS)

    # 训练模型
    start_time = time.time()
    best_acc = classifier.train(
        trainloader=trainloader,
        testloader=testloader,  # 使用测试集作为验证集
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    logging.info(f"最终最佳准确率: {best_acc:.2f}%")
    
    # 加载并验证保存的最佳模型
    if best_acc > 0:
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的最佳模型")
        logging.info("="*30)
        
        # 加载模型
        loaded_classifier = CoFrNetClassifier.load_model(MODEL_SAVE_PATH)
        
        # 验证模型性能
        verification_acc = loaded_classifier.evaluate(testloader)
        logging.info(f"加载后的模型在测试集上的验证精度: {verification_acc:.2f}%")
        
        # 检查验证结果
        assert abs(verification_acc - loaded_classifier.best_test_acc) < 1e-3, "验证失败！"
        logging.info("验证成功！模型可以被正确保存和复用。")
        
        # 计算模型参数数量
        total_params = sum(p.numel() for p in loaded_classifier.ensemble_model.parameters() if p.requires_grad)
        logging.info(f"模型总参数量: {total_params:,}")
    else:
        logging.info("未能成功训练和保存模型。")

2025-09-18 08:06:57,697 [INFO ]  实验超参数: {
    "input_dim": 3072,
    "output_dim": 10,
    "num_models_max": 100,
    "accuracy_target": 1.0,
    "epochs_per_model": 30,
    "shallow_depth_per_cofrnet": 2,
    "polynomial_degree": 3,
    "boosting_learning_rate": 0.1,
    "learning_rate_adam": 0.0001,
    "weight_decay": 1e-05,
    "early_stopping_patience": 10,
    "task_name": "CIFAR-10 Classification"
}
2025-09-18 08:06:57,698 [INFO ]  正在准备CIFAR-10数据集...


100%|██████████| 170M/170M [00:10<00:00, 15.7MB/s] 


2025-09-18 08:07:11,924 [INFO ]  数据集准备完毕。
2025-09-18 08:07:11,925 [INFO ]  CoFrNetClassifier 初始化完毕，将在 cuda 上运行。
2025-09-18 08:07:11,926 [INFO ]  ==============================
2025-09-18 08:07:11,927 [INFO ]    开始 CIFAR-10 分类任务训练 (选择性加入机制)
2025-09-18 08:07:11,927 [INFO ]  ==============================
2025-09-18 08:07:11,928 [INFO ]  --- 尝试添加第 1 个子模型 (总尝试次数: 1/100) ---
2025-09-18 08:08:39,261 [INFO ]      Epoch 010 | 训练损失: 2.2337 | Val Acc: 29.72%
2025-09-18 08:10:04,467 [INFO ]      Epoch 020 | 训练损失: 2.1950 | Val Acc: 31.55%
2025-09-18 08:11:27,511 [INFO ]      Epoch 030 | 训练损失: 2.1867 | Val Acc: 31.52%
2025-09-18 08:11:27,512 [INFO ]  候选模型训练完毕，正在评估其对整体性能的贡献...
2025-09-18 08:11:28,907 [INFO ]  加入候选模型后，测试集 Acc 为: 31.52%
2025-09-18 08:11:28,909 [INFO ]  当前最佳模型的测试集 Acc 为: 0.00%
2025-09-18 08:11:28,911 [INFO ]  *** 性能提升！采纳新模型。当前模型大小: 1, 新的最佳 Acc: 31.52% ***
2025-09-18 08:11:28,912 [INFO ]  --- 尝试添加第 2 个子模型 (总尝试次数: 2/100) ---
2025-09-18 08:12:53,502 [INFO ]      Epoch 010 | 训练损失: 2.1244 |